In [18]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn tensorflow boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists


In [20]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/hojjatk/mnist-dataset"

!unzip -q -o ./dataset.zip -d ./dataset/

/home/ec2-user/SageMaker


In [28]:
# Import pip packages
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import struct
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

In [22]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [23]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [50]:
from sagemaker.tensorflow import TensorFlow

estimator = TensorFlow(
    entry_point="train.py",
    framework_version="2.19.0",
    py_version="py312",
    instance_type="ml.m6i.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,
        'lr': 0.001,
    },
    metric_definitions = [
        {"Name": "train:loss", "Regex": r"loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"accuracy: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"val_loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"val_accuracy: ([0-9\.]+)"},
    ],
)

estimator.fit({"train": f's3://{bucket}/{prefix}/dataset.zip'})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: tensorflow-training-2026-07-01-07-59-00-458


2026-07-01 07:59:02 Starting - Starting the training job...
2026-07-01 07:59:33 Downloading - Downloading input data...
2026-07-01 07:59:53 Downloading - Downloading the training image...
2026-07-01 08:00:28 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-07-01 08:00:43.898191: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-01 08:00:43.930584: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate c

In [51]:
# Inference src source dir
!mkdir src
!cp train.py src/inference.py
!touch src/requirements.txt
!echo "numpy" >> src/requirements.txt
!echo "Pillow" >> src/requirements.txt
!echo "tensorflow" >> src/requirements.txt

In [52]:
# Create new model + Update Endpoint
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
    name=f'mnist-model-{int(time.time())}',
    entry_point = "inference.py",
    source_dir = "src"
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'mnist-endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker.tensorflow.model:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating model with name: mnist-model-1782893001


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/mnist-endpoint-configuration-1782893003',
 'ResponseMetadata': {'RequestId': '23f02538-224f-4fe0-8fac-e9e7812208d9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '23f02538-224f-4fe0-8fac-e9e7812208d9',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '120',
   'date': 'Wed, 01 Jul 2026 08:03:24 GMT'},
  'RetryAttempts': 0}}

In [53]:
# sm.create_endpoint(
#     EndpointName = "mnist-endpoint",
#     EndpointConfigName = new_config
# )

sm.update_endpoint(
    EndpointName = "mnist-endpoint",
    EndpointConfigName = new_config
)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/mnist-endpoint',
 'ResponseMetadata': {'RequestId': '88180005-26b4-4eb0-9410-2e82d85e916a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '88180005-26b4-4eb0-9410-2e82d85e916a',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '82',
   'date': 'Wed, 01 Jul 2026 08:03:34 GMT'},
  'RetryAttempts': 0}}

In [55]:
CLASSES = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
CLASSES

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [56]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'mnist-endpoint'
image_path = "9328.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 7, 'confidence': 0.23196931668405268}
7
